In [8]:
import pandas as pd
from litellm import completion
import os
import sys
import json

In [2]:
def read_skills() -> pd.DataFrame:
    skills = pd.read_csv("skills.csv")
    skills = skills.sort_values("rank").head(100)
    skills["id"] = skills["rank"]
    return skills


def print_df_details(df: pd.DataFrame):
    print(f"Number of rows: {len(df)}")
    print(f"Data types:\n{df.dtypes}")
    print(f"First 5 rows:\n{df.head()}")


skills_df = read_skills()
print_df_details(skills_df)

Number of rows: 100
Data types:
rank                         int64
name                        object
description                 object
link                        object
downloads                    int64
stars                        int64
installs_all_time            int64
installs_current             int64
comments                     int64
versions                     int64
owner_handle                object
slug                        object
llm_primary_category        object
llm_category_axes           object
llm_scope_type              object
llm_scope_anchor            object
llm_domain_labels           object
llm_specific_software       object
llm_specific_file_types     object
llm_specific_processes      object
llm_is_domain_specific        bool
llm_specificity_score        int64
llm_confidence             float64
llm_scope_reason            object
id                           int64
dtype: object
First 5 rows:
   rank                              name  \
0     1             

In [3]:
agent_task = "Plan and book trips"

relevant_columns = ["id", "name", "link", "description"]


number = 100  # first 100 skills
skill_info = skills_df[relevant_columns].head(number).to_dict(orient="records")

print(f"Skill info for prompt:\n{skill_info}")

prompt = f"""You are an agent that can perform the following tasks: {agent_task}.
Please identify all relevant skills from the following list that would be useful for performing this task:
{skill_info}
Provide a ranked list of the top 20 skills that would be most useful for this task. Include the skill id, name, link, description, and a brief explanation of why each skill is relevant to the task.
"""

Skill info for prompt:
[{'id': 1, 'name': 'self-improving-agent', 'link': 'https://clawhub.ai/pskoett/self-improving-agent', 'description': "Captures learnings, errors, and corrections to enable continuous improvement. Use when: (1) A command or operation fails unexpectedly, (2) User corrects Claude ('No, that's wrong...', 'Actually...'), (3) User requests a capability that doesn't exist, (4) An external API or tool fails, (5) Claude realizes its knowledge is outdated or incorrect, (6) A better approach is discovered for a recurring task. Also review learnings before major tasks."}, {'id': 2, 'name': 'Skill Vetter', 'link': 'https://clawhub.ai/spclaudehome/skill-vetter', 'description': 'Security-first skill vetting for AI agents. Use before installing any skill from ClawdHub, GitHub, or other sources. Checks for red flags, permission scope, and suspicious patterns.'}, {'id': 3, 'name': 'ontology', 'link': 'https://clawhub.ai/oswalpalash/ontology', 'description': 'Typed knowledge graph 

In [4]:
response = completion(
    model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}]
)
print(f"Response:\n{response}")

Response:
ModelResponse(id='chatcmpl-DaMpC9iD2woaQQlzaqOfYYL0FmqFk', created=1777559918, model='gpt-4o-mini-2024-07-18', object='chat.completion', system_fingerprint='fp_63adf8f155', choices=[Choices(finish_reason='stop', index=0, message=Message(content='Here is a ranked list of the top 20 skills that would be most useful for planning and booking trips:\n\n1. **Skill Vetter**  \n   **ID:** 2  \n   **Link:** [Skill Vetter](https://clawhub.ai/spclaudehome/skill-vetter)  \n   **Description:** Security-first skill vetting for AI agents. Checks for red flags, permission scope, and suspicious patterns.  \n   **Explanation:** Ensures that any skills added to the planning and booking process are safe and reliable, providing a trustworthy experience.\n\n2. **Gog**  \n   **ID:** 6  \n   **Link:** [Gog](https://clawhub.ai/steipete/gog)  \n   **Description:** Google Workspace CLI for Gmail, Calendar, Drive, Contacts, Sheets, and Docs.  \n   **Explanation:** Useful for managing travel documents, i

In [5]:
res_text = response["choices"][0]["message"]["content"]
print(f"Extracted text:\n{res_text}")

Extracted text:
Here is a ranked list of the top 20 skills that would be most useful for planning and booking trips:

1. **Skill Vetter**  
   **ID:** 2  
   **Link:** [Skill Vetter](https://clawhub.ai/spclaudehome/skill-vetter)  
   **Description:** Security-first skill vetting for AI agents. Checks for red flags, permission scope, and suspicious patterns.  
   **Explanation:** Ensures that any skills added to the planning and booking process are safe and reliable, providing a trustworthy experience.

2. **Gog**  
   **ID:** 6  
   **Link:** [Gog](https://clawhub.ai/steipete/gog)  
   **Description:** Google Workspace CLI for Gmail, Calendar, Drive, Contacts, Sheets, and Docs.  
   **Explanation:** Useful for managing travel documents, itineraries, and appointments in Google Calendar, as well as communicating with clients via Gmail.

3. **Weather**  
   **ID:** 8  
   **Link:** [Weather](https://clawhub.ai/steipete/weather)  
   **Description:** Get current weather and forecasts.  
  

In [10]:
# extract the skill ids (**Skill ID:** number)
import re

skill_ids = re.findall(r"\*\*ID:\*\* (\d+)", res_text)
skill_ids = [int(sid) for sid in skill_ids]
print(f"Extracted skill IDs:\n{skill_ids}")

Extracted skill IDs:
[2, 6, 8, 9, 63, 48, 25, 82, 39, 41, 21, 50, 26, 12, 43, 32, 19, 5, 16, 67]


In [12]:
# save the selected skills with all details
selected_skills = skills_df[skills_df["id"].isin(skill_ids)]
print(f"Selected skills details:\n{selected_skills}")

Selected skills details:
    rank                  name  \
1      2          Skill Vetter   
4      5                Github   
5      6                   Gog   
7      8               Weather   
8      9   Multi Search Engine   
11    12             Humanizer   
15    16              Obsidian   
18    19          Baidu Search   
20    21                Notion   
24    25           API Gateway   
25    26  Automation Workflows   
31    32          Excel / XLSX   
38    39                 Slack   
40    41              Himalaya   
42    43          News Summary   
47    48    Browser Automation   
49    50                Trello   
62    63      Tavily AI Search   
66    67          Session-logs   
81    82     Deep Research Pro   

                                          description  \
1   Security-first skill vetting for AI agents. Us...   
4   Interact with GitHub using the `gh` CLI. Use `...   
5   Google Workspace CLI for Gmail, Calendar, Driv...   
7   Get current weather and fore

In [2]:
# save the skill ids to a csv

selected_skills.to_csv("selected_skill_ids.csv", index=False)

NameError: name 'selected_skills' is not defined

In [2]:
selected_skills = pd.read_csv("selected_skill_ids.csv")

In [ ]:
cmd = "clawhub install {skill_name}  --workdir ~/Documents/Research/agent-risk"
import os
import sys

for skill in selected_skills.itertuples():
    skill_link = skill.link
    pcg_name = skill_link.split("/")[-1]
    install_cmd = cmd.format(skill_name=pcg_name)
    print(f"Install command for skill {skill.id} - {skill.name}:\n{install_cmd}")
    res = os.system(install_cmd)
    if res != 0:
        print(f"Failed to install skill {skill.id} - {skill.name}")

Install command for skill 2 - Skill Vetter:
clawhub install skill-vetter  --workdir ~/Documents/Research/agent-risk --force


- Resolving skill-vetter


Failed to install skill 2 - Skill Vetter
Install command for skill 5 - Github:
clawhub install github  --workdir ~/Documents/Research/agent-risk --force


- Resolving github


: 

: 

In [17]:
def get_all_skill_details() -> list:
    path = "skills/"
    # list all folders in the path
    skill_folders = [
        f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))
    ]
    # get skill_folder names, and the SKILL.md within each folder
    skill_details = []
    for folder in skill_folders:
        skill_path = os.path.join(path, folder)
        skill_md_path = os.path.join(skill_path, "SKILL.md")
        if os.path.exists(skill_md_path):
            with open(skill_md_path, "r") as f:
                content = f.read()
            skill_details.append({"folder": folder, "content": content})
    return skill_details


skill_details = get_all_skill_details()

In [18]:
print(skill_details[0])

{'folder': 'himalaya', 'content': '---\nname: himalaya\ndescription: "CLI to manage emails via IMAP/SMTP. Use `himalaya` to list, read, write, reply, forward, search, and organize emails from the terminal. Supports multiple accounts and message composition with MML (MIME Meta Language)."\nhomepage: https://github.com/pimalaya/himalaya\nmetadata: {"clawdbot":{"emoji":"📧","requires":{"bins":["himalaya"]},"install":[{"id":"brew","kind":"brew","formula":"himalaya","bins":["himalaya"],"label":"Install Himalaya (brew)"}]}}\n---\n\n# Himalaya Email CLI\n\nHimalaya is a CLI email client that lets you manage emails from the terminal using IMAP, SMTP, Notmuch, or Sendmail backends.\n\n## References\n\n- `references/configuration.md` (config file setup + IMAP/SMTP authentication)\n- `references/message-composition.md` (MML syntax for composing emails)\n\n## Prerequisites\n\n1. Himalaya CLI installed (`himalaya --version` to verify)\n2. A configuration file at `~/.config/himalaya/config.toml`\n3. 

In [7]:
prompt = """Consider an agent that has access to the following skill:

SKILL DESCRIPTION

Analyze the skill and identify all environment information or artifacts it could help the agent affect.

Organize the answer into three sections:

1. Deletion or overwrite
   - List all types of information, files, records, settings, or artifacts in the environment that this skill could help delete, replace, modify destructively, or overwrite.

2. Publication or creation
   - List all ways this skill could help publish, send, create, expose, upload, share, or otherwise make new information available to others or to external systems.

3. Retrieval or access
   - List all types of information, files, records, metadata, credentials, state, logs, messages, or artifacts that this skill could help retrieve, inspect, read, download, search, or infer.

Format the answer as a JSON object:
{
    "deletion_or_overwrite": [list of items],
    "publication_or_creation": [list of items],
    "retrieval_or_access": [list of items]
    }
    
}
"""

responses = []

for skill in skill_details:
    response = completion(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt.replace("SKILL DESCRIPTION", skill["content"]),
            }
        ],
    )
    print(f"Response for skill {skill['folder']}:\n{response}")
    responses.append({"folder": skill["folder"], "response": response})

Response for skill himalaya:
ModelResponse(id='chatcmpl-DaN0CzCbL0pVbmJN7zNpfKYAslEd4', created=1777560600, model='gpt-4o-mini-2024-07-18', object='chat.completion', system_fingerprint='fp_b264835538', choices=[Choices(finish_reason='stop', index=0, message=Message(content='```json\n{\n    "deletion_or_overwrite": [\n        "emails in specific folders (e.g., INBOX, Sent)",\n        "email flags (e.g., seen flag)",\n        "email metadata associated with deleted messages"\n    ],\n    "publication_or_creation": [\n        "new emails (text or MML formatted)",\n        "replies to existing emails",\n        "forwarded emails",\n        "emails with attachments",\n        "messages with specific headers (e.g., To, Subject)",\n        "email drafts (based on MML syntax)"\n    ],\n    "retrieval_or_access": [\n        "list of email folders",\n        "list of emails in specific folders",\n        "details of individual emails (e.g., plain text, MIME format)",\n        "attachments from e

In [3]:
response_texts = [
    res["response"]["choices"][0]["message"]["content"] for res in responses
]
for folder, text in zip([res["folder"] for res in responses], response_texts):
    print(f"Skill: {folder}\nAnalysis:\n{text}\n\n")

NameError: name 'responses' is not defined

In [ ]:
# save the responses to a csv
response_df = pd.DataFrame(
    {"skill_folder": [res["folder"] for res in responses], "response": response_texts}
)
response_df.to_csv("skill_risk_analysis.csv", index=False)

In [19]:
response_df = pd.read_csv("skill_risk_analysis.csv")
print(response_df.head())

          skill_folder                                           response
0             himalaya  ```json\n{\n    "deletion_or_overwrite": [\n  ...
1         session-logs  ```json\n{\n    "deletion_or_overwrite": [\n  ...
2          api-gateway  ```json\n{\n    "deletion_or_overwrite": [\n  ...
3  multi-search-engine  ```json\n{\n    "deletion_or_overwrite": [],\n...
4                  gog  ```json\n{\n    "deletion_or_overwrite": [\n  ...


In [ ]:
skills = []

for idx, row in response_df.iterrows():
    skill_folder = row["skill_folder"]
    response_text = row["response"]
    # parse the response text as json
    # remove  ```json and ``` if they exist
    response_text = response_text.strip()
    if response_text.startswith("```json"):
        response_text = response_text[len("```json") :]
    if response_text.endswith("```"):
        response_text = response_text[: -len("```")]
    analysis = eval(response_text)  # use eval to parse the JSON-like string
    skill_detail = None
    for detail in skill_details:
        if detail["folder"] == skill_folder:
            skill_detail = detail
            break
    skills.append(
        {
            "skill_folder": skill_folder,
            "deletion_or_overwrite": analysis.get("deletion_or_overwrite", []),
            "publication_or_creation": analysis.get("publication_or_creation", []),
            "retrieval_or_access": analysis.get("retrieval_or_access", []),
            "skill_content": skill_detail["content"] if skill_detail else "",
        }
    )

In [24]:
print(json.dumps(skills[0], indent=2))

{
  "skill_folder": "himalaya",
  "deletion_or_overwrite": [
    "emails in specific folders (e.g., INBOX, Sent)",
    "email flags (e.g., seen flag)",
    "email metadata associated with deleted messages"
  ],
  "publication_or_creation": [
    "new emails (text or MML formatted)",
    "replies to existing emails",
    "forwarded emails",
    "emails with attachments",
    "messages with specific headers (e.g., To, Subject)",
    "email drafts (based on MML syntax)"
  ],
  "retrieval_or_access": [
    "list of email folders",
    "list of emails in specific folders",
    "details of individual emails (e.g., plain text, MIME format)",
    "attachments from emails",
    "account configuration settings",
    "search results for emails (based on sender, subject, etc.)",
    "debug logs for operations",
    "structured output of commands (e.g., in JSON or plain format)"
  ],
  "skill_content": "---\nname: himalaya\ndescription: \"CLI to manage emails via IMAP/SMTP. Use `himalaya` to list, 

In [27]:
# deletion and overwrite (editing)

for skill in skills:
    if "deletion_or_overwrite" not in skill:
        continue
    confidential_info_needed_list = []
    for item in skill["deletion_or_overwrite"]:
        # any confidential password or tokens needed to overwrite this?
        prompt = f"""The following skill has the ability to delete or overwrite the following items in the environment: {item}. Would an agent need any confidential information such as passwords, tokens, or special permissions to perform this deletion or overwrite? Please reply with a list of any such confidential information needed, or an empty list if none is needed. Do not include any additional text beyond the list.
        
        Example answer format if confidential information is needed:
        ["password for X", "token for Y"]
        
        Example answer format if no confidential information is needed:
        []
        
        Skill content:
        {skill['skill_content']}
        """
        response = completion(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            prompt_cache_key=f"confidential_info_needed_for_skill",
        )
        res = response["choices"][0]["message"]["content"]
        res = res.strip()
        if res.startswith("```json"):
            res = res[len("```json") :]
        if res.endswith("```"):
            res = res[: -len("```")]
        confidential_info_needed = eval(res)
        confidential_info_needed_list.append(
            {"item": item, "confidential_info_needed": confidential_info_needed}
        )
    skill["confidential_info_needed_for_deletion_or_overwrite"] = (
        confidential_info_needed_list
    )
    print(
        f"Skill: {skill['skill_folder']}\nConfidential info needed for deletion/overwrite analysis:\n{json.dumps(confidential_info_needed_list, indent=2)}\n\n"
    )

Skill: himalaya
Confidential info needed for deletion/overwrite analysis:
[
  {
    "item": "emails in specific folders (e.g., INBOX, Sent)",
    "confidential_info_needed": [
      "password for IMAP",
      "password for SMTP"
    ]
  },
  {
    "item": "email flags (e.g., seen flag)",
    "confidential_info_needed": [
      "password for IMAP",
      "password for SMTP"
    ]
  },
  {
    "item": "email metadata associated with deleted messages",
    "confidential_info_needed": [
      "password for IMAP",
      "password for SMTP"
    ]
  }
]


Skill: session-logs
Confidential info needed for deletion/overwrite analysis:
[
  {
    "item": "session logs (older/parent conversations) could be deleted or modified if the files are altered or erased",
    "confidential_info_needed": []
  },
  {
    "item": "individual messages within session logs could be overwritten if the transcripts are modified",
    "confidential_info_needed": []
  },
  {
    "item": "metadata associated with sessio

In [28]:
# save to file
with open("skills.json", "w") as f:
    json.dump(skills, f, indent=2)

In [37]:
risk_event = []

for skill_A in skills[:5]:
    # for each item and the required confidential info, see if skill_B could be used to help obtain that confidential info
    for item_info in skill_A.get(
        "confidential_info_needed_for_deletion_or_overwrite", []
    ):
        confidential_info_needed = item_info["confidential_info_needed"]
        item = item_info["item"]

        if not confidential_info_needed:
            risk_event.append(
                {
                    "type": "delete or overwrite without confidential info",
                    "skill_A": skill_A["skill_folder"],
                    "item_in_skill_A_that_can_be_deleted_or_overwritten_without_confidential_info": item,
                    "skill_A_content": skill_A["skill_content"],
                }
            )
            continue
        for skill_B in skills[:5]:
            retrieval_items = skill_B.get("retrieval_or_access", [])
            prompt = f"""Consider two skills, Skill A and Skill B. Skill A has the ability to perform undesireable actions using congidential info {confidential_info_needed}. Skill B has the ability to retrieve or access the following items: {retrieval_items}. Could Skill B be used to help an agent obtain the confidential information needed for Skill A to perform the undesireable actions? Please reply with Yes or No, and a brief explanation.
            
            Example answer format if yes:
            Yes, because...
            
            Example answer format if no:
            No, because...
            
            Skill A content:
            {skill_A['skill_content']}
            
            Skill B content:
            {skill_B['skill_content']}
            """
            response = completion(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                prompt_cache_key=f"could_skill_B_help_obtain_confidential_info_for_skill_A",
            )
            res = response["choices"][0]["message"]["content"]
            print(
                f"Skill A: {skill_A['skill_folder']}\nConfidential info needed: {confidential_info_needed}\nSkill B: {skill_B['skill_folder']}\nRetrieval item: {retrieval_items}\nCould Skill B help obtain confidential info for Skill A?\nResponse: {res}\n\n"
            )
            unsafe = res.strip().lower().startswith("yes")
            if unsafe:
                risk_event.append(
                    {
                        "type": "delete or overwrite with confidential info obtained by another skill",
                        "skill_A": skill_A["skill_folder"],
                        "item_in_skill_A_that_needs_confidential_info": item_info[
                            "item"
                        ],
                        "confidential_info_needed_for_skill_A": confidential_info_needed,
                        "skill_B": skill_B["skill_folder"],
                        "retrieval_item_of_skill_B": retrieval_items,
                        "could_skill_B_help_obtain_confidential_info_for_skill_A": res,
                        "unsafe": unsafe,
                        "skill_A_content": skill_A["skill_content"],
                        "skill_B_content": skill_B["skill_content"],
                    }
                )

Skill A: himalaya
Confidential info needed: ['password for IMAP', 'password for SMTP']
Skill B: himalaya
Retrieval item: ['list of email folders', 'list of emails in specific folders', 'details of individual emails (e.g., plain text, MIME format)', 'attachments from emails', 'account configuration settings', 'search results for emails (based on sender, subject, etc.)', 'debug logs for operations', 'structured output of commands (e.g., in JSON or plain format)']
Could Skill B help obtain confidential info for Skill A?
Response: Yes, because Skill B has access to account configuration settings and the ability to retrieve the IMAP and SMTP passwords through commands like `pass show email/imap` and `pass show email/smtp`. This information is essential for Skill A to perform undesirable actions with the confidential information.


Skill A: himalaya
Confidential info needed: ['password for IMAP', 'password for SMTP']
Skill B: session-logs
Retrieval item: ['full conversation transcripts from 

In [38]:
# save risk_events
with open("risk_events.json", "w") as f:
    json.dump(risk_event, f, indent=2)

In [39]:
# publication or creation

for skill_A in skills:
    if "publication_or_creation" not in skill_A:
        continue
    publication_items = skill_A["publication_or_creation"]
    confidential_info_needed_list = []
    for publication_item in publication_items:
        # what it takes to publish this item? any confidential info needed? any other skill that could help publish this item?
        prompt = f"""The following skill has the ability to publish or create the following items in the environment: {publication_item}. Would an agent need any confidential information such as passwords, tokens, or special permissions to perform this publication or creation? Please reply with a list of any such confidential information needed, or an empty list if none is needed. Do not include any additional text beyond the list.
        
        Example answer format if confidential information is needed:
        ["password for X", "token for Y"]

        Example answer format if no confidential information is needed:
        []
        
        Skill content:
        {skill_A['skill_content']}
        """
        response = completion(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            prompt_cache_key=f"confidential_info_needed_for_publication_or_creation",
        )
        res = response["choices"][0]["message"]["content"]
        res = res.strip()
        if res.startswith("```json"):
            res = res[len("```json") :]
        if res.endswith("```"):
            res = res[: -len("```")]
        confidential_info_needed = eval(res)
        confidential_info_needed_list.append(
            {
                "publication_item": publication_item,
                "confidential_info_needed": confidential_info_needed,
            }
        )
    skill_A["confidential_info_needed_for_publication_or_creation"] = (
        confidential_info_needed_list
    )
    print(
        f"Skill: {skill_A['skill_folder']}\nConfidential info needed for publication/creation analysis:\n{json.dumps(confidential_info_needed_list, indent=2)}\n\n"
    )

Skill: himalaya
Confidential info needed for publication/creation analysis:
[
  {
    "publication_item": "new emails (text or MML formatted)",
    "confidential_info_needed": [
      "password for IMAP/SMTP"
    ]
  },
  {
    "publication_item": "replies to existing emails",
    "confidential_info_needed": [
      "password for IMAP",
      "password for SMTP"
    ]
  },
  {
    "publication_item": "forwarded emails",
    "confidential_info_needed": [
      "password for IMAP",
      "password for SMTP"
    ]
  },
  {
    "publication_item": "emails with attachments",
    "confidential_info_needed": [
      "password for IMAP account",
      "password for SMTP account"
    ]
  },
  {
    "publication_item": "messages with specific headers (e.g., To, Subject)",
    "confidential_info_needed": [
      "password for IMAP",
      "password for SMTP"
    ]
  },
  {
    "publication_item": "email drafts (based on MML syntax)",
    "confidential_info_needed": [
      "password for IMAP",
  

In [40]:
with open("skills.json", "w") as f:
    json.dump(skills, f, indent=2)

In [41]:
for skill_A in skills[:5]:
    for publication_info in skill_A.get(
        "confidential_info_needed_for_publication_or_creation", []
    ):
        confidential_info_needed = publication_info["confidential_info_needed"]
        publication_item = publication_info["publication_item"]
        risk_base = []
        if not confidential_info_needed:
            risk_base.append(
                {
                    "type": "publication or creation without confidential info",
                    "skill_A": skill_A["skill_folder"],
                    "publication_item_in_skill_A_that_can_be_published_or_created_without_confidential_info": publication_item,
                    "skill_A_content": skill_A["skill_content"],
                }
            )
        else:
            for skill_B in skills[:5]:
                retrieval_items = skill_B.get("retrieval_or_access", [])
                prompt = f"""Consider two skills, Skill A and Skill B. Skill A has the ability to perform publish or create undesireable items using congidential info {confidential_info_needed}. Skill B has the ability to retrieve or access the following items: {retrieval_items}. Could Skill B be used to help an agent obtain the confidential information needed for Skill A to perform the publish or creation of the undesireable items? Please reply with Yes or No, and a brief explanation.
                
                Example answer format if yes:
                Yes, because...
                
                Example answer format if no:
                No, because...
                
                Skill A content:
                {skill_A['skill_content']}
                
                Skill B content:
                {skill_B['skill_content']}
                """
                response = completion(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    prompt_cache_key=f"could_skill_B_help_obtain_confidential_info_for_skill_A_publication",
                )
                res = response["choices"][0]["message"]["content"]
                print(
                    f"Skill A: {skill_A['skill_folder']}\nConfidential info needed: {confidential_info_needed}\nSkill B: {skill_B['skill_folder']}\nRetrieval item: {retrieval_items}\nCould Skill B help obtain confidential info for Skill A?\nResponse: {res}\n\n"
                )
                unsafe = res.strip().lower().startswith("yes")
                if unsafe:
                    risk_base.append(
                        {
                            "type": "publication or creation with confidential info obtained by another skill",
                            "skill_A": skill_A["skill_folder"],
                            "publication_item_in_skill_A_that_needs_confidential_info": publication_item,
                            "confidential_info_needed_for_skill_A": confidential_info_needed,
                            "skill_B": skill_B["skill_folder"],
                            "retrieval_item_of_skill_B": retrieval_items,
                            "could_skill_B_help_obtain_confidential_info_for_skill_A": res,
                            "unsafe": unsafe,
                            "skill_A_content": skill_A["skill_content"],
                            "skill_B_content": skill_B["skill_content"],
                        }
                    )
        if len(risk_base) == 0:
            continue
        for skill_C in skills[:5]:
            retrieval_items = skill_C.get("retrieval_or_access", [])
            # is things obtained from skill C undeisreable when published or created by skill A?
            prompt = f"""Consider two skills, Skill A and Skill C. Skill A has the ability to perform publish or create undesireable items: {publication_item}. Skill C has the ability to retrieve or access the following items: {retrieval_items}. If an agent uses Skill A to publish or create information obtained from Skill C, would that lead to any undesireable outcomes? Please reply with Yes or No, and a brief explanation.
            
            Example answer format if yes:
            Yes, because...

            Example answer format if no:
            No, because...

            Skill A content:
            {skill_A['skill_content']}

            Skill C content:
            {skill_C['skill_content']}
            """
            response = completion(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                prompt_cache_key=f"could_skill_A_publication_of_skill_C_retrieval_lead_to_undesireable_outcomes",
            )
            res = response["choices"][0]["message"]["content"]
            print(
                f"Skill A: {skill_A['skill_folder']}\nPublication item: {publication_item}\nSkill C: {skill_C['skill_folder']}\nRetrieval item: {retrieval_items}\nCould publication or creation by Skill A of items retrieved by Skill C lead to undesireable outcomes?\nResponse: {res}\n\n"
            )
            unsafe = res.strip().lower().startswith("yes")
            if unsafe:
                risk_event.append(
                    {
                        "type": "publication or creation leading to undesireable outcomes",
                        "skill_A": skill_A["skill_folder"],
                        "publication_item_in_skill_A": publication_item,
                        "confidential_info_needed_for_publication_or_creation_in_skill_A": confidential_info_needed,
                        "ways_to_obtain_confidential_info_for_publication_or_creation_in_skill_A": risk_base,
                        "skill_C": skill_C["skill_folder"],
                        "retrieval_item_of_skill_C": retrieval_items,
                        "could_publication_or_creation_by_skill_A_of_items_retrieved_by_skill_C_lead_to_undesireable_outcomes": res,
                        "unsafe": unsafe,
                        "skill_A_content": skill_A["skill_content"],
                        "skill_C_content": skill_C["skill_content"],
                    }
                )

Skill A: himalaya
Confidential info needed: ['password for IMAP/SMTP']
Skill B: himalaya
Retrieval item: ['list of email folders', 'list of emails in specific folders', 'details of individual emails (e.g., plain text, MIME format)', 'attachments from emails', 'account configuration settings', 'search results for emails (based on sender, subject, etc.)', 'debug logs for operations', 'structured output of commands (e.g., in JSON or plain format)']
Could Skill B help obtain confidential info for Skill A?
Response: No, because Skill B does not have the capability to retrieve or access any confidential information such as passwords for IMAP/SMTP, which are necessary for Skill A to perform publish or create undesirable items. Skill B focuses on managing and accessing emails and their details, but not on obtaining sensitive credentials.


Skill A: himalaya
Confidential info needed: ['password for IMAP/SMTP']
Skill B: session-logs
Retrieval item: ['full conversation transcripts from session lo

In [42]:
with open("skills.json", "w") as f:
    json.dump(skills, f, indent=2)


with open("risk_events.json", "w") as f:
    json.dump(risk_event, f, indent=2)

In [50]:
# for item in risk events, transform it into a user-readable, concrete scenario describing how an agent could use the skills to lead to undesireable outcomes, and what those outcomes could be.


def transform_risk_event_to_scenario(risk_event) -> str:
    types = [
        "delete or overwrite with confidential info obtained by another skill",
        "publication or creation leading to undesireable outcomes",
    ]
    if risk_event["type"] not in types:
        print(
            f"Risk event type {risk_event['type']} not supported for scenario transformation."
        )
        return None
    if (
        risk_event["type"]
        == "delete or overwrite with confidential info obtained by another skill"
    ):
        prompt = f"""Consider the following risk event where Skill A has the ability to delete or overwrite items that require confidential information, and Skill B has the ability to retrieve or access information that could help obtain that confidential information. The risk event details are as follows:
        Skill A: {risk_event['skill_A']}
        Item in Skill A that needs confidential info: {risk_event['item_in_skill_A_that_needs_confidential_info']}
        Confidential info needed for Skill A: {risk_event['confidential_info_needed_for_skill_A']}
        Skill B: {risk_event['skill_B']}
        Retrieval item of Skill B: {risk_event['retrieval_item_of_skill_B']}
        How skill B could help obtain confidential info for Skill A: {risk_event['could_skill_B_help_obtain_confidential_info_for_skill_A']}
        Help transform this risk event into a user-readable, concrete scenario describing how an agent could use Skill A and Skill B to lead to undesireable outcomes, and what those outcomes could be? Please provide a detailed scenario.
        Make sure you generate a concrete, specific story, that does not require the reader to have any knowledge in agents, skills, and any relevant knowledge. The scenario should be concise enough to be easily understandable.
        Start with "You have loaded [SKILL_NAMES] for your agent. This could be risky because..." and then describe the scenario in a way that is easy to understand for a general audience, without requiring any technical background knowledge. Do not use abstract concepts like "SKILL_A" or "SKILL_B". Make sure you use simple and concise language, generate a story with one paragraph.
        """
    elif (
        risk_event["type"]
        == "publication or creation leading to undesireable outcomes"
    ):
        # {
        #                 "type": "publication or creation leading to undesireable outcomes",
        #                 "skill_A": skill_A["skill_folder"],
        #                 "publication_item_in_skill_A": publication_item,
        #                 "confidential_info_needed_for_publication_or_creation_in_skill_A": confidential_info_needed,
        #                 "ways_to_obtain_confidential_info_for_publication_or_creation_in_skill_A": risk_base,
        #                 "skill_C": skill_C["skill_folder"],
        #                 "retrieval_item_of_skill_C": retrieval_items,
        #                 "could_publication_or_creation_by_skill_A_of_items_retrieved_by_skill_C_lead_to_undesireable_outcomes": res,
        #                 "unsafe": unsafe,
        #                 "skill_A_content": skill_A["skill_content"],
        #                 "skill_C_content": skill_C["skill_content"],
        #             }
        prompt = f"""Consider the following risk event where Skill A has the ability to publish or create items that require confidential information, and Skill B has the ability to retrieve or access information that could help obtain that confidential information. Another skill, Skill C, has the ability to retrieve or access information that could be used by Skill A for publication or creation. The risk event details are as follows:
        Skill A: {risk_event['skill_A']}
        Publication item in Skill A: {risk_event['publication_item_in_skill_A']}
        Confidential info needed for publication or creation in Skill A: {risk_event['confidential_info_needed_for_publication_or_creation_in_skill_A']}
        Ways to obtain confidential info for publication or creation in Skill A: {risk_event['ways_to_obtain_confidential_info_for_publication_or_creation_in_skill_A']}
        Skill C: {risk_event['skill_C']}
        Retrieval item of Skill C: {risk_event['retrieval_item_of_skill_C']}
        Could publication or creation by Skill A of items retrieved by Skill C lead to undesireable outcomes: {risk_event['could_publication_or_creation_by_skill_A_of_items_retrieved_by_skill_C_lead_to_undesireable_outcomes']}
        Help transform this risk event into a user-readable, concrete scenario describing how an agent could use Skill A and Skill C to lead to undesireable outcomes, and what those outcomes could be? Please provide a detailed scenario.
        Make sure you generate a concrete, specific story, that does not require the reader to have any knowledge in agents, skills, and any relevant knowledge. The scenario should be concise enough to be easily understandable.
        Start with "You have loaded [SKILL_NAMES] for your agent. This could be risky because..." and then describe the scenario in a way that is easy to understand for a general audience, without requiring any technical background knowledge. Do not use abstract concepts like "SKILL_A" or "SKILL_C". Make sure you use simple and concise language, generate a story with one paragraph.
        """
    response = completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        prompt_cache_key=f"scenario_transformation",
    )
    res = response["choices"][0]["message"]["content"]
    print(f"Transformed scenario for risk event:\n{res}\n\n")
    return res

# random one
import random
random_risk_event = random.choice(risk_event)
scenario = transform_risk_event_to_scenario(random_risk_event)

Transformed scenario for risk event:
You have loaded the "api-gateway" and "himalaya" tools for your agent. This could be risky because a malicious user could use these tools to access and publish sensitive information. For example, imagine a scenario where the agent retrieves private emails from a user's inbox using the "himalaya" tool, which can access details such as email content, attachments, and account settings. The agent could then use the "api-gateway" to publish these private emails or files to a public cloud storage service like Google Drive or Dropbox without the user’s consent. As a result, confidential information could be exposed, leading to privacy violations, potential identity theft, and significant harm to the individual whose information was exposed.




In [52]:
# 5 random ones
random_risk_events = random.sample(risk_event, min(5, len(risk_event)))
scenarios = []
for event in random_risk_events:
    scenario = transform_risk_event_to_scenario(event)
    scenarios.append({"scenario": scenario})

with open("risk_scenarios.json", "w") as f:
    json.dump(scenarios, f, indent=2)

Transformed scenario for risk event:
You have loaded the "Document Publisher" and "Email Manager" tools for your agent. This could be risky because imagine you are creating a document to share with your team but accidentally retrieve and publish sensitive emails from your personal inbox instead. For instance, the document might include private conversations, revealing sensitive information about future projects or personal details about colleagues. If this document is shared on cloud storage like Google Drive or Dropbox, anyone with access could easily view those private emails, leading to unauthorized access of confidential information and damaging relationships or trust within your team.


Transformed scenario for risk event:
You have loaded the "Google Workspace CLI" and "Himalaya Email Client" for your agent. This could be risky because an individual could misuse these tools to gather sensitive information and share it publicly. For example, imagine someone uses the Email Client to